# 评估验证集、测试集的准确率

## 模型定义

In [1]:
fold_num = 1
family_fold_type = 'GTB'

import argparse
import textwrap
import os
import time
import dgl
import torch
import torch.nn as nn
import torch.nn.functional as F
from dgl.data import tu
from model.encoder import DiffPool
from livelossplot import PlotLosses
from sklearn.metrics import f1_score
import shutil
import pandas as pd
import os
import numpy as np
import dgl
from dgl import DGLGraph
from dgl.data.utils import save_graphs, load_graphs, save_info, load_info
import torch

class customreaddata:
    """
    A custom dataset class to read graph data from specified text files for prediction.
    Assumes the following files exist in raw_dir under a subdirectory named 'name':
    - {name}_A.txt (edge list)
    - {name}_graph_indicator.txt (which graph each node belongs to)
    - {name}_graph_labels.txt (labels for each graph) - This might be dummy for prediction
    - {name}_node_attributes.txt (features for each node)

    Parameters
    ----------
    name : str
        Name of the dataset directory and prefix for files (e.g., 'GTmining').
    raw_dir : str
        Path to the directory containing the dataset folder.
    """

    def __init__(self, name, raw_dir):
        self.name = name
        self.raw_dir = raw_dir
        self.save_dir = os.path.join(raw_dir, name) # Use raw_dir as base, create save path inside
        os.makedirs(self.save_dir, exist_ok=True) # Ensure save directory exists for potential caching

        # Initialize attributes that will be set by process()
        self.graph_lists = []
        self.graph_labels = []
        self.item_ids = []  # 新增：存储每个图的item_id
        self.max_num_node = 0
        self.num_labels = None # May not be relevant for prediction

        # Process the raw data files
        self.process()

    def _file_path(self, category):
        """Constructs the path to a specific data file."""
        return os.path.join(self.raw_dir, f"{self.name}_{category}.txt")

    @staticmethod
    def _idx_from_zero(idx_tensor):
        """Adjusts indices to be 0-based."""
        # Assuming node and graph indices in your files are 1-based.
        # If they are already 0-based, this step is unnecessary or needs adjustment.
        # Check the first few lines of your GTmining_graph_indicator.txt to confirm.
        # If they are 0-based, remove this function or make it a no-op.
        # For now, assuming 1-based as per TUDataset standard.
        min_val = np.min(idx_tensor)
        if min_val == 0:
             # If already 0-based, return as is or handle accordingly
             # This might be the case, adjust logic if needed
             print(f"Warning: Indices in file seem to be 0-based (min={min_val}). Proceeding assuming 0-based.")
             return idx_tensor
        else:
             # Standard 1-based to 0-based conversion
             return idx_tensor - 1 # More standard than subtracting min if known to start at 1

    def process(self):
        """
        Loads data from text files and constructs a list of DGLGraphs.
        """
        print(f"Processing custom dataset: {self.name}")

        # --- 1. Load Edge List ---
        print(f"Loading edges from {self._file_path('A')}")
        # Load edges, assuming 1-based indexing initially
        edge_data_raw = np.genfromtxt(self._file_path("A"), delimiter=",", dtype=int)
        if edge_data_raw.ndim == 1:
            # If only one edge, reshape to (1, 2)
            edge_data_raw = edge_data_raw.reshape(1, -1)
        # Convert to 0-based indices
        edge_data_0_based = self._idx_from_zero(edge_data_raw)
        # DGL expects source and destination arrays
        src_nodes = edge_data_0_based[:, 0]
        dst_nodes = edge_data_0_based[:, 1]

        # --- 2. Load Graph Indicator (which graph each node belongs to) ---
        print(f"Loading graph indicators from {self._file_path('graph_indicator')}")
        node_graph_ids_raw = np.loadtxt(self._file_path("graph_indicator"), dtype=int)
        # Convert graph IDs to 0-based indices
        node_graph_ids = self._idx_from_zero(node_graph_ids_raw)
        num_total_nodes_in_file = len(node_graph_ids_raw)

        # --- 3. Load Graph Labels (might be dummy) ---
        print(f"Loading graph labels from {self._file_path('graph_labels')}")
        try:
            graph_labels_raw = np.loadtxt(self._file_path("graph_labels"), dtype=int)
            # Convert graph labels to 0-based indices if needed, though for classification
            # they often represent class IDs starting from 0 or 1. Adjust if necessary.
            # For prediction, these might just be placeholders.
            self.graph_labels = graph_labels_raw # Keep original values for now, adjust if necessary
            self.num_labels = max(self.graph_labels) + 1 if len(self.graph_labels) > 0 else 0
        except FileNotFoundError:
            print(f"Warning: Graph labels file {self._file_path('graph_labels')} not found. Using dummy labels (e.g., 0).")
            num_graphs_in_file = len(set(node_graph_ids))
            self.graph_labels = np.zeros(num_graphs_in_file, dtype=int) # Dummy labels
            self.num_labels = 1 # Or set to None if not applicable

        # --- 4. Load Item IDs (新增逻辑) ---
        print(f"Loading item IDs from {self._file_path('itemID')}")
        try:
            # 读取itemID文件，支持整数或字符串类型的ID
            self.item_ids = np.loadtxt(self._file_path("itemID"), dtype=str).tolist()
            
            # 验证item_id数量是否与图数量匹配
            num_graphs_in_file = len(set(node_graph_ids))
            if len(self.item_ids) != num_graphs_in_file:
                raise ValueError(
                    f"Number of item IDs ({len(self.item_ids)}) does not match number of graphs ({num_graphs_in_file})."
                )
        except FileNotFoundError:
            print(f"Warning: Item ID file {self._file_path('itemID')} not found. Using graph index as item ID.")
            num_graphs_in_file = len(set(node_graph_ids))
            self.item_ids = [str(i) for i in range(num_graphs_in_file)]  # 使用索引作为默认ID


        # --- 4. Load Node Attributes ---
        print(f"Loading node attributes from {self._file_path('node_attributes')}")
        try:
            node_attributes = np.loadtxt(self._file_path("node_attributes"), delimiter=",")
            if node_attributes.ndim == 1:
                # If features are 1D (one feature per node), reshape to (num_nodes, 1)
                node_attributes = np.expand_dims(node_attributes, axis=1)
            print(f"Loaded node attributes with shape: {node_attributes.shape}")
            if node_attributes.shape[0] != num_total_nodes_in_file:
                 raise ValueError(f"Number of rows in node_attributes ({node_attributes.shape[0]}) does not match number of nodes indicated by graph_indicator ({num_total_nodes_in_file}).")
        except FileNotFoundError:
            print(f"Warning: Node attributes file {self._file_path('node_attributes')} not found. Graphs will have no node features (ndata['feat'] will not be set initially).")
            node_attributes = None


        # --- 5. Create a Base Graph with All Nodes and Edges ---
        # This graph contains all nodes from all graphs, connected by the provided edges.
        num_nodes_in_base_graph = int(np.max(src_nodes)) + 1 if len(src_nodes) > 0 else 0
        # Ensure num_nodes includes any isolated nodes that might only appear in graph_indicator
        num_nodes_in_base_graph = max(num_nodes_in_base_graph, num_total_nodes_in_file)

        if num_nodes_in_base_graph == 0:
            print("Warning: No nodes or edges found in the data files.")
            self.graph_lists = []
            return

        base_graph = dgl.graph(([], []), num_nodes=num_nodes_in_base_graph)
        base_graph.add_edges(src_nodes, dst_nodes)

        # Assign node attributes to the base graph if available
        if node_attributes is not None:
            base_graph.ndata['feat'] = torch.tensor(node_attributes, dtype=torch.float32)

        # --- 6. Split the Base Graph into Individual Graphs ---
        self.graph_lists = []
        self.max_num_node = 0

        num_expected_graphs = len(set(node_graph_ids))
        print(f"Found {num_expected_graphs} graphs based on graph_indicator.")

        for graph_id in range(num_expected_graphs):
            # Find the nodes belonging to the current graph (graph_id)
            node_mask = (node_graph_ids == graph_id)
            node_indices_for_graph = np.where(node_mask)[0] # Get 0-based indices of nodes in this graph

            if len(node_indices_for_graph) == 0:
                print(f"Warning: Graph ID {graph_id} has no nodes according to graph_indicator.")
                # Create an empty graph for this ID
                g_sub = dgl.graph(([], []), num_nodes=0)
                # Add a dummy feature tensor if original had features, though shape might be tricky for 0 nodes
                # Often, empty graphs might need special handling downstream.
                # For now, just create the empty graph.
            else:
                # Extract the subgraph corresponding to these nodes
                g_sub = base_graph.subgraph(node_indices_for_graph)

                # The subgraph's nodes have new IDs (0, 1, ...). The original features are preserved based on the subgraph operation.
                # If node_attributes was loaded, 'feat' is already in g_sub.ndata.
                # Check if 'feat' exists, otherwise features were not available.
                if 'feat' not in g_sub.ndata:
                     print(f"  Graph {graph_id}: No node features available.")


            self.graph_lists.append(g_sub)

            if g_sub.num_nodes() > self.max_num_node:
                self.max_num_node = g_sub.num_nodes()

        print(f"Successfully processed {len(self.graph_lists)} graphs.")
        print(f"Max number of nodes in a single graph: {self.max_num_node}")


    def __getitem__(self, idx):
        """
        Gets the graph and its label at the given index.

        Parameters
        ---------
        idx : int
            The sample index.

        Returns
        -------
        dgl.DGLGraph
            The graph object, potentially with node features in `ndata['feat']`.
        torch.Tensor
            The label tensor for the graph (could be dummy for prediction).
        """
        if idx < 0 or idx >= len(self):
             raise IndexError(f"Index {idx} is out of range for dataset with {len(self)} items.")
        g = self.graph_lists[idx]
        label = torch.tensor(self.graph_labels[idx], dtype=torch.int64) if self.graph_labels is not None else torch.tensor(0, dtype=torch.int64) # Return a dummy label if not set
        item_id = self.item_ids[idx]  # 新增：获取对应索引的item_id

        return g, label, item_id

    def __len__(self):
        """
        Returns the number of graphs in the dataset.
        """
        return len(self.graph_lists)

    @property
    def num_classes(self):
        """Returns the number of classes (uses num_labels)."""
        return int(self.num_labels) if self.num_labels is not None else 0 # Return 0 if not set
    
    def statistics(self):
        """
        返回数据集的三个关键统计信息，适配你的调用格式：
        input_dim, label_dim, max_num_node
        """
        # 1. 节点特征维度（如果没有特征则返回 0）
        if len(self.graph_lists) > 0 and 'feat' in self.graph_lists[0].ndata:
            input_dim = self.graph_lists[0].ndata['feat'].shape[1]
        else:
            input_dim = 0

        # 2. 标签维度（类别数量）
        label_dim = self.num_classes

        # 3. 最大节点数（你已经在 process 里算好了）
        max_num_node = self.max_num_node

        return input_dim, label_dim, max_num_node
    



def prepare_data(dataset, shuffle=False, prog_args=None, custom_batch_size=None):
    """
    preprocess TU dataset according to DiffPool's paper setting and load dataset into dataloader
    """
    if custom_batch_size is None:
        return dgl.dataloading.GraphDataLoader(
            dataset,
            batch_size=prog_args.batch_size,
            shuffle=shuffle,
            num_workers=prog_args.n_worker,
        )
    else:
        return dgl.dataloading.GraphDataLoader(
            dataset,
            batch_size=custom_batch_size,
            shuffle=shuffle,
            num_workers=prog_args.n_worker,
        )


print("{:=^100}".format(f'fold num is : {fold_num}, family type is : {family_fold_type}'))

print("{:=^100}".format('prog_args'))
prog_args = argparse.Namespace(dataset=f'GTmining_6_6_{family_fold_type}_fold{fold_num}', pool_ratio=0.10, num_pool=1, cuda=1, lr=1.0, clip=float("inf"),
                               batch_size=128, epoch=500, n_worker=10, gc_per_block=3, aggregator_type="meanpool",
                               dropout=0.00, method="diffpool", bn=True, bias=True, save_dir=f"./model_param_alldata/",
                               load_epoch=-1, data_mode="default", linkpred=False, hidden_dim=64, embedding_dim=64, family_fold_type=family_fold_type)
print( textwrap.fill(str(prog_args), width=100))

print("{:=^100}".format('加载数据'))
dataset_train = customreaddata(name="GTmining",
                                    raw_dir=f'../data/dl_data/{family_fold_type}_alldata_id/fold{fold_num}/train/')
dataset_validation = customreaddata(name="GTmining",
                                   raw_dir=f'../data/dl_data/{family_fold_type}_alldata_id/fold{fold_num}/validation/')
dataset_test = customreaddata(name="GTmining",
                                   raw_dir=f'../data/dl_data/{family_fold_type}_alldata_id/fold{fold_num}/test/')
train_dataloader = prepare_data(dataset_train, shuffle=True, prog_args=prog_args)
validation_dataloader = prepare_data(dataset_validation, shuffle=False, prog_args=prog_args)
test_dataloader = prepare_data(dataset_test, shuffle=False, prog_args=prog_args)

input_dim_train, label_dim_train, max_num_node_train = dataset_train.statistics()
input_dim_validation, label_dim_validation, max_num_node_validation = dataset_validation.statistics()
input_dim_test, label_dim_test, max_num_node_test = dataset_test.statistics()
max_num_node = max([max_num_node_train, max_num_node_validation, max_num_node_test])
input_dim = input_dim_train
label_dim = label_dim_train
print("++++++++++ STATISTICS ABOUT THE DATASET ++++++++++")
print("dataset feature dimension is", input_dim_train)
print("dataset label dimension is", label_dim_train)
print("the max num node is", max_num_node)
print("number of graphs is", len(dataset_train) + len(dataset_validation)+ len(dataset_test))

hidden_dim = prog_args.hidden_dim  # used to be 64
embedding_dim = prog_args.embedding_dim

assign_dim = int(max_num_node * prog_args.pool_ratio)
print("++++++++++MODEL STATISTICS++++++++")
print("model hidden dim is", hidden_dim)
print("model embedding dim for graph instance embedding", embedding_dim)
print("initial batched pool graph dim is", assign_dim)
activation = F.relu

if family_fold_type == 'GTA':
    graph_label_dict = {'UDP-Glc': 0, 'UDP-GlcNAc': 1, 'UDP-GlcA': 2,
                        'UDP-Gal': 3, 'UDP-GalNAc': 4,
                        'UDP-Xyl': 5, 'GDP-Man': 6,
                        'dTDP-Rha': 7, 'Other': 8}
elif family_fold_type == 'GTB':
    graph_label_dict = {'UDP-Glc': 0, 'UDP-GlcNAc': 1, 'UDP-GlcA': 2,
                        'UDP-Gal': 3, 'UDP-GalNAc': 4,
                        'UDP-Xyl': 5, 'GDP-Man': 6, 'GDP-Fuc': 7,
                        'dTDP-Rha': 8, 'Other': 9}
else:
    raise ValueError(f"Invalid family_fold_type: '{prog_args.family_fold_type}'. Valid options are 'GTA' and 'GTB'.")
df_cluster = pd.read_excel(f'../data/cluster/{family_fold_type}/dataseat_split_{fold_num}.xlsx')
df_cluster = df_cluster.loc[df_cluster['Dataset']=='train']
df_cluster.reset_index(drop=True, inplace=True)
custom_loss_weight = []
total_sample = df_cluster.shape[0]
for x in graph_label_dict.keys():
    df_x = df_cluster.loc[df_cluster['Activate']==x]
    df_x.reset_index(drop=True, inplace=True)
    x_sample = df_x.shape[0]
    custom_loss_weight.append(total_sample/x_sample)

assert len(custom_loss_weight) == label_dim_train, 'Wrong custom loss weight, please check what happen.'

# initialize model
model = DiffPool(
    input_dim,
    hidden_dim,
    embedding_dim,
    label_dim,
    activation,
    prog_args.gc_per_block,
    prog_args.dropout,
    prog_args.num_pool,
    prog_args.linkpred,
    prog_args.batch_size,
    prog_args.aggregator_type,
    assign_dim,
    prog_args.pool_ratio,
    custom_loss_weight
)
print("model init finished")
print("MODEL:::::::", prog_args.method)
if prog_args.cuda:
    model = model.cuda()




/home/admin123/software/miniconda3/envs/GTmining_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


===============================fold num is : 1, family type is : GTB================================
=============================================prog_args==============================================
Namespace(dataset='GTmining_6_6_GTB_fold1', pool_ratio=0.1, num_pool=1, cuda=1, lr=1.0, clip=inf,
batch_size=128, epoch=500, n_worker=10, gc_per_block=3, aggregator_type='meanpool', dropout=0.0,
method='diffpool', bn=True, bias=True, save_dir='./model_param_alldata/', load_epoch=-1,
data_mode='default', linkpred=False, hidden_dim=64, embedding_dim=64, family_fold_type='GTB')
================================================加载数据================================================
Processing custom dataset: GTmining
Loading edges from ../data/dl_data/GTB_alldata_id/fold1/train/GTmining_A.txt
Loading graph indicators from ../data/dl_data/GTB_alldata_id/fold1/train/GTmining_graph_indicator.txt
Loading graph labels from ../data/dl_data/GTB_alldata_id/fold1/train/GTmining_graph_labels.txt
Loading i

In [2]:
# 先过一遍train_dataloader，让模型中的一些参数先初始化一下
model.train()
for batch_idx, (batch_graph, graph_labels, item_ids) in enumerate(train_dataloader):
    for key, value in batch_graph.ndata.items():
        batch_graph.ndata[key] = value.float()
    graph_labels = graph_labels.long()
    if torch.cuda.is_available():
        batch_graph = batch_graph.to(torch.cuda.current_device())
        graph_labels = graph_labels.cuda()
    ypred = model(batch_graph)
    loss = model.loss(ypred, graph_labels)
    loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), max_norm=prog_args.clip)
    model.zero_grad()



## 开始评估

In [4]:
import pandas as pd

# GTA_best_epoch = {
#         1: 367, 2: 764, 3: 631, 4: 570,
#         5: 451, 6: 473, 7: 704, 8: 292,
#         9: 364, 10: 353
#     }
GTA_best_epoch = {
        1: 367, 2: 764, 3: 631, 4: 594,
        5: 451, 6: 473, 7: 704, 8: 832,
        9: 364, 10: 353
    }
GTB_best_epoch = {
        1: 206, 2: 245, 3: 152, 4: 141,
        5: 225, 6: 137, 7: 119, 8: 216,
        9: 205, 10: 143
    }

print(GTA_best_epoch)
print(GTB_best_epoch)


{1: 367, 2: 764, 3: 631, 4: 594, 5: 451, 6: 473, 7: 704, 8: 832, 9: 364, 10: 353}
{1: 206, 2: 245, 3: 152, 4: 141, 5: 225, 6: 137, 7: 119, 8: 216, 9: 205, 10: 143}


In [ ]:
from collections import defaultdict, Counter

if family_fold_type == 'GTA':
    best_epoch_dict = GTA_best_epoch
elif family_fold_type == 'GTB':
    best_epoch_dict = GTB_best_epoch
else:
    raise ValueError('SSSSSB')



for fold_num_predict in range(1, len(list(best_epoch_dict.keys()))+1):
    
    fold_num = fold_num_predict
    dataset_validation = customreaddata(name="GTmining",
                                   raw_dir=f'../data/dl_data/{family_fold_type}_alldata_id/fold{fold_num}/validation/')
    validation_dataloader = prepare_data(dataset_validation, shuffle=False, prog_args=prog_args)


    
    predict_result = defaultdict(list)
    data_label = defaultdict(list)

    epoch_predict = best_epoch_dict[fold_num_predict]

    begin_time = time.time()
    print("\nEPOCH ###### Fold {} Epoch {} ######".format(fold_num_predict, epoch_predict))
    if epoch_predict is not None and prog_args.save_dir is not None:
        model.load_state_dict(
            torch.load(
                prog_args.save_dir
                + "/"
                + prog_args.dataset
                + "/model.iter-"
                + "{:04d}".format(epoch_predict), weights_only=True
            )
        )


    model.eval()
    with torch.no_grad():
        val_pred_indi = torch.tensor([], device='cuda')
        val_label_indi = torch.tensor([], device='cuda')
        for batch_idx, (batch_graph, graph_labels, item_id) in enumerate(validation_dataloader):
            for key, value in batch_graph.ndata.items():
                batch_graph.ndata[key] = value.float()
            graph_labels = graph_labels.long()
            if torch.cuda.is_available():
                batch_graph = batch_graph.to(torch.cuda.current_device())
                graph_labels = graph_labels.cuda()

            # 拿标签
            temp = batch_graph

            protein_ids = item_id


            ypred = model(batch_graph)
            indi = torch.argmax(ypred, dim=1)
            for x in range(0, len(indi)):
                predict_result[protein_ids[x]].append(int(indi[x].cpu()))
                data_label[protein_ids[x]].append(int(graph_labels[x].cpu()))

            
    elapsed_time = time.time() - begin_time
    print("epoch {:.4f} with epoch time {:.4f} s".format(epoch_predict, elapsed_time))

    df = pd.DataFrame(columns=['protein_id', 'predict_label_list', 'true_label_list'])
    all_count = 0
    correct_count = 0
    for x in predict_result.keys():
        # 创建单行DataFrame
        new_row = pd.DataFrame({
            'protein_id': [x],  # 注意要加中括号，保持数组形式
            'predict_label_list': [predict_result[x][0]],
            'true_label_list': [data_label[x][0]]
        })
        # 使用concat追加行
        df = pd.concat([df, new_row], ignore_index=True)
        if predict_result[x][0] == data_label[x][0]:
            correct_count += 1
        all_count += 1
        # print(f"protein_id: {x}, predict label list: {predict_result[x]}, true label list: {data_label[x]}")

    print(f"Accuracy: {correct_count}/{all_count} = {correct_count/all_count:.4f}")
    # df.to_excel(f'./result/alldata_itemID_accuracy/{family_fold_type}_fold{fold_num}_validation.xlsx', index=False)

    # break

Processing custom dataset: GTmining
Loading edges from ../data/dl_data/GTB_alldata_id/fold1/validation/GTmining_A.txt
Loading graph indicators from ../data/dl_data/GTB_alldata_id/fold1/validation/GTmining_graph_indicator.txt
Loading graph labels from ../data/dl_data/GTB_alldata_id/fold1/validation/GTmining_graph_labels.txt
Loading item IDs from ../data/dl_data/GTB_alldata_id/fold1/validation/GTmining_itemID.txt
Loading node attributes from ../data/dl_data/GTB_alldata_id/fold1/validation/GTmining_node_attributes.txt
Loaded node attributes with shape: (1526610, 7)
Found 3774 graphs based on graph_indicator.
Successfully processed 3774 graphs.
Max number of nodes in a single graph: 630

EPOCH ###### Fold 1 Epoch 206 ######


/home/admin123/software/miniconda3/envs/GTmining_env/lib/python3.10/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


epoch 206.0000 with epoch time 9.2534 s
Accuracy: 3423/3774 = 0.9070
Processing custom dataset: GTmining
Loading edges from ../data/dl_data/GTB_alldata_id/fold2/validation/GTmining_A.txt
Loading graph indicators from ../data/dl_data/GTB_alldata_id/fold2/validation/GTmining_graph_indicator.txt
Loading graph labels from ../data/dl_data/GTB_alldata_id/fold2/validation/GTmining_graph_labels.txt
Loading item IDs from ../data/dl_data/GTB_alldata_id/fold2/validation/GTmining_itemID.txt
Loading node attributes from ../data/dl_data/GTB_alldata_id/fold2/validation/GTmining_node_attributes.txt
Loaded node attributes with shape: (1518958, 7)
Found 3815 graphs based on graph_indicator.
Successfully processed 3815 graphs.
Max number of nodes in a single graph: 630

EPOCH ###### Fold 2 Epoch 245 ######
epoch 245.0000 with epoch time 8.5206 s
Accuracy: 3589/3815 = 0.9408
Processing custom dataset: GTmining
Loading edges from ../data/dl_data/GTB_alldata_id/fold3/validation/GTmining_A.txt
Loading graph 

In [ ]:
from collections import defaultdict, Counter

if family_fold_type == 'GTA':
    best_epoch_dict = GTA_best_epoch
elif family_fold_type == 'GTB':
    best_epoch_dict = GTB_best_epoch
else:
    raise ValueError('SSSSSB')



for fold_num_predict in range(1, len(list(best_epoch_dict.keys()))+1):
    
    fold_num = fold_num_predict
    test_validation = customreaddata(name="GTmining",
                                   raw_dir=f'../data/dl_data/{family_fold_type}_alldata_id/fold{fold_num}/test/')
    test_dataloader = prepare_data(test_validation, shuffle=False, prog_args=prog_args)


    
    predict_result = defaultdict(list)
    data_label = defaultdict(list)

    epoch_predict = best_epoch_dict[fold_num_predict]

    begin_time = time.time()
    print("\nEPOCH ###### Fold {} Epoch {} ######".format(fold_num_predict, epoch_predict))
    if epoch_predict is not None and prog_args.save_dir is not None:
        model.load_state_dict(
            torch.load(
                prog_args.save_dir
                + "/"
                + prog_args.dataset
                + "/model.iter-"
                + "{:04d}".format(epoch_predict), weights_only=True
            )
        )


    model.eval()
    with torch.no_grad():
        val_pred_indi = torch.tensor([], device='cuda')
        val_label_indi = torch.tensor([], device='cuda')
        for batch_idx, (batch_graph, graph_labels, item_id) in enumerate(test_dataloader):
            for key, value in batch_graph.ndata.items():
                batch_graph.ndata[key] = value.float()
            graph_labels = graph_labels.long()
            if torch.cuda.is_available():
                batch_graph = batch_graph.to(torch.cuda.current_device())
                graph_labels = graph_labels.cuda()

            # 拿标签
            temp = batch_graph

            protein_ids = item_id


            ypred = model(batch_graph)
            indi = torch.argmax(ypred, dim=1)
            for x in range(0, len(indi)):
                predict_result[protein_ids[x]].append(int(indi[x].cpu()))
                data_label[protein_ids[x]].append(int(graph_labels[x].cpu()))

            
    elapsed_time = time.time() - begin_time
    print("epoch {:.4f} with epoch time {:.4f} s".format(epoch_predict, elapsed_time))

    df = pd.DataFrame(columns=['protein_id', 'predict_label_list', 'true_label_list'])
    all_count = 0
    correct_count = 0
    for x in predict_result.keys():
        # 创建单行DataFrame
        new_row = pd.DataFrame({
            'protein_id': [x],  # 注意要加中括号，保持数组形式
            'predict_label_list': [predict_result[x][0]],
            'true_label_list': [data_label[x][0]]
        })
        # 使用concat追加行
        df = pd.concat([df, new_row], ignore_index=True)
        if predict_result[x][0] == data_label[x][0]:
            correct_count += 1
        all_count += 1
        # print(f"protein_id: {x}, predict label list: {predict_result[x]}, true label list: {data_label[x]}")

    print(f"Accuracy: {correct_count}/{all_count} = {correct_count/all_count:.4f}")
    # df.to_excel(f'./result/alldata_itemID_accuracy/{family_fold_type}_fold{fold_num}_test.xlsx', index=False)

    # break

Processing custom dataset: GTmining
Loading edges from ../data/dl_data/GTB_alldata_id/fold1/test/GTmining_A.txt
Loading graph indicators from ../data/dl_data/GTB_alldata_id/fold1/test/GTmining_graph_indicator.txt
Loading graph labels from ../data/dl_data/GTB_alldata_id/fold1/test/GTmining_graph_labels.txt
Loading item IDs from ../data/dl_data/GTB_alldata_id/fold1/test/GTmining_itemID.txt
Loading node attributes from ../data/dl_data/GTB_alldata_id/fold1/test/GTmining_node_attributes.txt
Loaded node attributes with shape: (858448, 7)
Found 2168 graphs based on graph_indicator.
Successfully processed 2168 graphs.
Max number of nodes in a single graph: 592

EPOCH ###### Fold 1 Epoch 206 ######
epoch 206.0000 with epoch time 5.3062 s
Accuracy: 1766/2168 = 0.8146
Processing custom dataset: GTmining
Loading edges from ../data/dl_data/GTB_alldata_id/fold2/test/GTmining_A.txt
Loading graph indicators from ../data/dl_data/GTB_alldata_id/fold2/test/GTmining_graph_indicator.txt
Loading graph label

In [24]:
predict_result

defaultdict(list,
            {'GT2_ABE58641_1': [0],
             'GT2_ABJ00437_1': [0],
             'GT2_ABR89677_1': [8],
             'GT2_ABU77545_1': [0],
             'GT2_ACS85370_1': [0],
             'GT2_ACT06459_1': [0],
             'GT2_ACT13553_1': [0],
             'GT2_ACZ76433_1': [0],
             'GT2_ADM99013_1': [8],
             'GT2_ADO48987_1': [0],
             'GT2_AEB59565_1': [0],
             'GT2_AEF23164_1': [0],
             'GT2_AEF44945_1': [0],
             'GT2_AEI80165_1': [0],
             'GT2_AEJ05989_1': [0],
             'GT2_AEN64325_1': [0],
             'GT2_AER33204_1': [8],
             'GT2_AEX52834_1': [0],
             'GT2_AFC87597_1': [0],
             'GT2_AFI90893_1': [0],
             'GT2_AFM34089_1': [0],
             'GT2_AFM59404_1': [0],
             'GT2_AFN78719_1': [0],
             'GT2_AFY17523_1': [0],
             'GT2_AGA75806_1': [0],
             'GT2_AGB78652_1': [0],
             'GT2_AGB82078_1': [0],
          

In [25]:
data_label

defaultdict(list,
            {'GT2_ABE58641_1': [0],
             'GT2_ABJ00437_1': [0],
             'GT2_ABR89677_1': [0],
             'GT2_ABU77545_1': [0],
             'GT2_ACS85370_1': [0],
             'GT2_ACT06459_1': [0],
             'GT2_ACT13553_1': [0],
             'GT2_ACZ76433_1': [0],
             'GT2_ADM99013_1': [0],
             'GT2_ADO48987_1': [0],
             'GT2_AEB59565_1': [0],
             'GT2_AEF23164_1': [0],
             'GT2_AEF44945_1': [0],
             'GT2_AEI80165_1': [0],
             'GT2_AEJ05989_1': [0],
             'GT2_AEN64325_1': [0],
             'GT2_AER33204_1': [0],
             'GT2_AEX52834_1': [0],
             'GT2_AFC87597_1': [0],
             'GT2_AFI90893_1': [0],
             'GT2_AFM34089_1': [0],
             'GT2_AFM59404_1': [0],
             'GT2_AFN78719_1': [0],
             'GT2_AFY17523_1': [0],
             'GT2_AGA75806_1': [0],
             'GT2_AGB78652_1': [0],
             'GT2_AGB82078_1': [0],
          

# 评估验证集测试集和训练集的序列相似性

In [ ]:
'''
下面是代码
'''

# from Bio import Align
# from Bio.Seq import Seq
# import Bio
# from Bio import SeqIO
# import multiprocessing
# from tqdm import tqdm
# from multiprocessing import Pool
# import pandas as pd

# def calculate_protein_similarity(seq1, seq2):
#     """
#     计算两条蛋白质序列的相似性（兼容不同Biopython版本）
#     :param seq1: 第一条蛋白质序列（字符串/Seq对象）
#     :param seq2: 第二条蛋白质序列（字符串/Seq对象）
#     :return: 比对结果、一致性比例、相似性比例
#     """
#     # 1. 初始化比对器，设置为蛋白质序列比对
#     aligner = Align.PairwiseAligner()
#     aligner.mode = 'global'  # 全局比对
#     aligner.substitution_matrix = Align.substitution_matrices.load("BLOSUM62")
#     # 用默认缺口罚分（无需手动设置）

#     # 2. 执行序列比对，取最优比对结果
#     alignments = aligner.align(seq1, seq2)
#     best_aln = alignments[0]

#     # 3. 兼容不同版本的序列提取方式（核心修复点）
#     # 将原始序列转为字符串，方便后续比对
#     seq1_str = str(seq1)
#     seq2_str = str(seq2)
    
#     # 从Alignment对象中提取比对的位置信息
#     # best_aln.aligned[0] 是第一条序列的比对区间，best_aln.aligned[1] 是第二条的
#     aln_pos1 = best_aln.aligned[0]
#     aln_pos2 = best_aln.aligned[1]
    
#     # 构建比对后的序列（补全缺口'-'）
#     aln_seq1 = []
#     aln_seq2 = []
#     pos1 = pos2 = 0  # 原始序列的指针
    
#     for (start1, end1), (start2, end2) in zip(aln_pos1, aln_pos2):
#         # 处理前面的缺口
#         gap1 = start1 - pos1
#         gap2 = start2 - pos2
#         if gap1 > 0:
#             aln_seq1.extend(['-'] * gap1)
#             aln_seq2.extend(seq2_str[pos2:pos2+gap1])
#             pos2 += gap1
#         if gap2 > 0:
#             aln_seq2.extend(['-'] * gap2)
#             aln_seq1.extend(seq1_str[pos1:pos1+gap2])
#             pos1 += gap2
        
#         # 添加匹配的序列片段
#         aln_seq1.extend(seq1_str[start1:end1])
#         aln_seq2.extend(seq2_str[start2:end2])
#         pos1, pos2 = end1, end2
    
#     # 处理末尾的缺口
#     if pos1 < len(seq1_str):
#         aln_seq1.extend(seq1_str[pos1:])
#         aln_seq2.extend(['-'] * (len(seq1_str) - pos1))
#     if pos2 < len(seq2_str):
#         aln_seq2.extend(seq2_str[pos2:])
#         aln_seq1.extend(['-'] * (len(seq2_str) - pos2))
    
#     # 转为字符串
#     aln_seq1 = ''.join(aln_seq1)
#     aln_seq2 = ''.join(aln_seq2)
    
#     # 4. 计算一致性（Identity）
#     match_count = sum(1 for a, b in zip(aln_seq1, aln_seq2) if a == b and a != '-')
#     valid_length = sum(1 for a, b in zip(aln_seq1, aln_seq2) if a != '-' or b != '-')
#     identity = match_count / valid_length if valid_length > 0 else 0.0

#     # 5. 计算相似性（Similarity）
#     blosum62 = Align.substitution_matrices.load("BLOSUM62")
#     similarity_count = 0
#     for a, b in zip(aln_seq1, aln_seq2):
#         if a == '-' or b == '-':
#             continue
#         if blosum62[a][b] > 0:
#             similarity_count += 1
#     similarity = similarity_count / valid_length if valid_length > 0 else 0.0

#     # 构造易读的比对结果字符串
#     aln_result_str = f"{aln_seq1}\n{aln_seq2}\n得分：{best_aln.score:.2f}"
#     return aln_result_str, identity, similarity

# def a_batch_task(ecosp_id, ecosp_seq, gtmining):
#     max_identity = 0
#     for gtmining_id, gtmining_seq in gtmining.items():
#         _, identity, _ = calculate_protein_similarity(ecosp_seq, gtmining_seq)
#         if identity > max_identity:
#             max_identity = identity
#     return ecosp_id, max_identity

# def unpack_batch_task(args):
#     """辅助函数：解包参数元组并调用a_batch_task"""
#     return a_batch_task(*args)


# # ------------------- 测试示例 -------------------
# if __name__ == "__main__":
#     for fold in range(1, 11):
#         # Read the file and split sequences into two dictionaries
#         test = {}
#         train_validation = {}
#         temp_dict = {}

#         for record in SeqIO.parse("./GTA.fasta", "fasta"):
#             temp_dict[record.id] = str(record.seq)

#         df = pd.read_excel(f'./cluster/GTA_alldata/dataseat_split_{fold}.xlsx')
#         for i in range(0, df.shape[0]):
#             id_ = df['GenBank'][i]
#             split = df['Dataset'][i]
#             if split == 'test':
#                 test[id_] = temp_dict[id_]
#             else:
#                 train_validation[id_] = temp_dict[id_]

#         # 构造参数元组列表（每个元组包含两个参数）
#         task_args = [(ecosp_id, seq, train_validation) for ecosp_id, seq in test.items()]

#         # 创建进程池执行任务（用具名函数替代lambda）
#         with multiprocessing.Pool(processes=180) as pool:
#             # 使用具名的unpack_batch_task函数，可被pickle序列化
#             identitys = list(tqdm(
#                 pool.imap(unpack_batch_task, task_args),  # 替换lambda为具名函数
#                 total=len(test),
#                 desc="计算序列相似性"  # 可选：给进度条加描述，更清晰
#             ))

#         df = pd.DataFrame({
#             "test_id": [item[0] for item in identitys],
#             "max_identity": [item[1] for item in identitys]
#         })

#         df.to_excel(f"./GTA_test_vs_train_validation_identity_fold{fold}.xlsx", index=False)






In [ ]:
'''
下面是代码
'''

from Bio import Align
from Bio.Seq import Seq
import Bio
from Bio import SeqIO
import multiprocessing
from tqdm import tqdm
from multiprocessing import Pool
import pandas as pd

def calculate_protein_similarity(seq1, seq2):
    """
    计算两条蛋白质序列的相似性（兼容不同Biopython版本）
    :param seq1: 第一条蛋白质序列（字符串/Seq对象）
    :param seq2: 第二条蛋白质序列（字符串/Seq对象）
    :return: 比对结果、一致性比例、相似性比例
    """
    # 1. 初始化比对器，设置为蛋白质序列比对
    aligner = Align.PairwiseAligner()
    aligner.mode = 'global'  # 全局比对
    aligner.substitution_matrix = Align.substitution_matrices.load("BLOSUM62")
    # 用默认缺口罚分（无需手动设置）

    # 2. 执行序列比对，取最优比对结果
    alignments = aligner.align(seq1, seq2)
    best_aln = alignments[0]

    # 3. 兼容不同版本的序列提取方式（核心修复点）
    # 将原始序列转为字符串，方便后续比对
    seq1_str = str(seq1)
    seq2_str = str(seq2)
    
    # 从Alignment对象中提取比对的位置信息
    # best_aln.aligned[0] 是第一条序列的比对区间，best_aln.aligned[1] 是第二条的
    aln_pos1 = best_aln.aligned[0]
    aln_pos2 = best_aln.aligned[1]
    
    # 构建比对后的序列（补全缺口'-'）
    aln_seq1 = []
    aln_seq2 = []
    pos1 = pos2 = 0  # 原始序列的指针
    
    for (start1, end1), (start2, end2) in zip(aln_pos1, aln_pos2):
        # 处理前面的缺口
        gap1 = start1 - pos1
        gap2 = start2 - pos2
        if gap1 > 0:
            aln_seq1.extend(['-'] * gap1)
            aln_seq2.extend(seq2_str[pos2:pos2+gap1])
            pos2 += gap1
        if gap2 > 0:
            aln_seq2.extend(['-'] * gap2)
            aln_seq1.extend(seq1_str[pos1:pos1+gap2])
            pos1 += gap2
        
        # 添加匹配的序列片段
        aln_seq1.extend(seq1_str[start1:end1])
        aln_seq2.extend(seq2_str[start2:end2])
        pos1, pos2 = end1, end2
    
    # 处理末尾的缺口
    if pos1 < len(seq1_str):
        aln_seq1.extend(seq1_str[pos1:])
        aln_seq2.extend(['-'] * (len(seq1_str) - pos1))
    if pos2 < len(seq2_str):
        aln_seq2.extend(seq2_str[pos2:])
        aln_seq1.extend(['-'] * (len(seq2_str) - pos2))
    
    # 转为字符串
    aln_seq1 = ''.join(aln_seq1)
    aln_seq2 = ''.join(aln_seq2)
    
    # 4. 计算一致性（Identity）
    match_count = sum(1 for a, b in zip(aln_seq1, aln_seq2) if a == b and a != '-')
    valid_length = sum(1 for a, b in zip(aln_seq1, aln_seq2) if a != '-' or b != '-')
    identity = match_count / valid_length if valid_length > 0 else 0.0

    # 5. 计算相似性（Similarity）
    blosum62 = Align.substitution_matrices.load("BLOSUM62")
    similarity_count = 0
    for a, b in zip(aln_seq1, aln_seq2):
        if a == '-' or b == '-':
            continue
        if blosum62[a][b] > 0:
            similarity_count += 1
    similarity = similarity_count / valid_length if valid_length > 0 else 0.0

    # 构造易读的比对结果字符串
    aln_result_str = f"{aln_seq1}\n{aln_seq2}\n得分：{best_aln.score:.2f}"
    return aln_result_str, identity, similarity

def a_batch_task(ecosp_id, ecosp_seq, gtmining):
    max_identity = 0
    for gtmining_id, gtmining_seq in gtmining.items():
        _, identity, _ = calculate_protein_similarity(ecosp_seq, gtmining_seq)
        if identity > max_identity:
            max_identity = identity
    return ecosp_id, max_identity

def unpack_batch_task(args):
    """辅助函数：解包参数元组并调用a_batch_task"""
    return a_batch_task(*args)


# ------------------- 测试示例 -------------------

test = {}
train_validation = {}
temp_dict = {}
for record in SeqIO.parse("./predict_data/GTB.fasta", "fasta"):
    temp_dict[record.id] = str(record.seq)

for fold in tqdm(range(1, 11)):
    df = pd.read_excel(f'./predict_data/cluster/GTB_alldata/dataseat_split_{fold}.xlsx')
    for i in range(0, df.shape[0]):
        id_ = df['GenBank'][i]
        split = df['Dataset'][i]
        if split == 'test':
            test[id_] = temp_dict[id_]
        else:
            train_validation[id_] = temp_dict[id_]

predict_dict = {}
for record in SeqIO.parse("./predict_data/YjiC_like.fasta", "fasta"):
    predict_dict[record.id] = str(record.seq)

for id_ in predict_dict.keys():
    id_, max_identity = a_batch_task(id_, predict_dict[id_], train_validation)
    print(f"Protein ID: {id_}, Max Identity with GTmining Train/Validation: {max_identity:.4f}")
    # # 构造参数元组列表（每个元组包含两个参数）
    # task_args = [(ecosp_id, seq, train_validation) for ecosp_id, seq in test.items()]

    # # 创建进程池执行任务（用具名函数替代lambda）
    # with multiprocessing.Pool(processes=180) as pool:
    #     # 使用具名的unpack_batch_task函数，可被pickle序列化
    #     identitys = list(tqdm(
    #         pool.imap(unpack_batch_task, task_args),  # 替换lambda为具名函数
    #         total=len(test),
    #         desc="计算序列相似性"  # 可选：给进度条加描述，更清晰
    #     ))

    # df = pd.DataFrame({
    #     "test_id": [item[0] for item in identitys],
    #     "max_identity": [item[1] for item in identitys]
    # })

    # df.to_excel(f"./GTA_test_vs_train_validation_identity_fold{fold}.xlsx", index=False)







100%|██████████| 10/10 [00:18<00:00,  1.83s/it]


Protein ID: YjiC, Max Identity with GTmining Train/Validation: 0.9924
Protein ID: BsGT1, Max Identity with GTmining Train/Validation: 0.5777
Protein ID: BaGT, Max Identity with GTmining Train/Validation: 0.5504
Protein ID: BgGT, Max Identity with GTmining Train/Validation: 0.7488
Protein ID: BpGT, Max Identity with GTmining Train/Validation: 0.9924
Protein ID: BamGT, Max Identity with GTmining Train/Validation: 0.5463
Protein ID: BssGT, Max Identity with GTmining Train/Validation: 0.5561


# 结合准确率和相似性进行分析

In [10]:
import pandas as pd
from collections import defaultdict

fold_tyep = 'GTB'
result_true = defaultdict(int)
result_false = defaultdict(int)
for fold_tyep in ['GTA', 'GTB']:
    for fold in range(1, 11):
        df_accuracy = pd.read_excel(f'./result/alldata_itemID_accuracy/{fold_tyep}_fold{fold}_test.xlsx')
        accuracy_dict = defaultdict(bool)
        for i in range(0, df_accuracy.shape[0]):
            protein_id = df_accuracy['protein_id'][i]
            predict_label = df_accuracy['predict_label_list'][i]
            true_label = df_accuracy['true_label_list'][i]
            if predict_label == true_label:
                accuracy_dict[protein_id] = True
            else:
                accuracy_dict[protein_id] = False

        df_identify = pd.read_excel(f'./result/alldata_itemID_identify/{fold_tyep}_test_vs_train_validation_identity_fold{fold}.xlsx')
        for i in range(0, df_identify.shape[0]):
            protein_id = df_identify['test_id'][i]
            max_identity = df_identify['max_identity'][i]
            if max_identity <= 0.1:
                if accuracy_dict[protein_id]:
                    result_true['0_10'] += 1
                else:
                    result_false['0_10'] += 1
            elif max_identity <= 0.2:
                if accuracy_dict[protein_id]:
                    result_true['10_20'] += 1
                else:
                    result_false['10_20'] += 1
            elif max_identity <= 0.3:
                if accuracy_dict[protein_id]:
                    result_true['20_30'] += 1
                else:
                    result_false['20_30'] += 1
            elif max_identity <= 0.4:
                if accuracy_dict[protein_id]:
                    result_true['30_40'] += 1
                else:
                    result_false['30_40'] += 1
            elif max_identity <= 0.5:
                if accuracy_dict[protein_id]:
                    result_true['40_50'] += 1
                else:
                    result_false['40_50'] += 1
            elif max_identity <= 0.6:
                if accuracy_dict[protein_id]:
                    result_true['50_60'] += 1
                else:
                    result_false['50_60'] += 1
            elif max_identity <= 0.7:
                if accuracy_dict[protein_id]:
                    result_true['60_70'] += 1
                else:
                    result_false['60_70'] += 1
            elif max_identity <= 0.8:
                if accuracy_dict[protein_id]:
                    result_true['70_80'] += 1
                else:
                    result_false['70_80'] += 1
            elif max_identity <= 0.9:
                if accuracy_dict[protein_id]:
                    result_true['80_90'] += 1
                else:
                    result_false['80_90'] += 1
            elif max_identity <= 1.0:
                if accuracy_dict[protein_id]:
                    result_true['90_100'] += 1
                else:
                    result_false['90_100'] += 1



    # break

for key in result_true.keys():
    print(f"Identity range {key}: True count = {result_true[key]}, False count = {result_false[key]}, Accuracy = {result_true[key]/(result_true[key]+result_false[key]):.4f}")





Identity range 50_60: True count = 5035, False count = 880, Accuracy = 0.8512
Identity range 60_70: True count = 9898, False count = 1218, Accuracy = 0.8904
Identity range 90_100: True count = 1626, False count = 213, Accuracy = 0.8842
Identity range 80_90: True count = 1818, False count = 189, Accuracy = 0.9058
Identity range 70_80: True count = 3736, False count = 394, Accuracy = 0.9046
Identity range 30_40: True count = 2574, False count = 1399, Accuracy = 0.6479
Identity range 40_50: True count = 3836, False count = 696, Accuracy = 0.8464
Identity range 20_30: True count = 337, False count = 183, Accuracy = 0.6481


In [15]:
protein_id

'GT95_QKY15055_1'

In [9]:
df_identify

,test_id,max_identity
0,GT84_QEN02151_1,0.585987
1,GT84_UVC10461_1,0.678201
2,GT84_AHG93687_1,0.630137
3,GT84_AUN36818_1,0.616438
4,GT84_QJR34711_1,0.619672
...,...,...
1311,GT2_WNG92670_1,0.813636
1312,GT2_WRO43863_1,0.933014
1313,GT111_QPK94470_1,0.416667
1314,GT95_CCO14965_1,0.409396


In [5]:
df_accuracy

,protein_id,predict_label_list,true_label_list
0,GT84_QEN02151_1,0,0
1,GT84_UVC10461_1,0,0
2,GT84_AHG93687_1,0,0
3,GT84_AUN36818_1,0,0
4,GT84_QJR34711_1,0,0
...,...,...,...
1311,GT2_WNG92670_1,8,8
1312,GT2_WRO43863_1,8,8
1313,GT111_QPK94470_1,0,8
1314,GT95_CCO14965_1,8,8


# 局部结构比对
- 选取底物范围6埃的残基进行比对
- 计算局部结构的相似性（适用USAlign完成，以RMSD为衡量标准）

## 提取训练集、验证集、测试集的局部结构

In [1]:
# GTA
# 路径：/home/admin123/work/GTmining/data/structures/GTA/
# GTB
# 路径：/home/admin123/work/GTmining/data/structures/GTB/

import os
from tqdm import tqdm
from Bio.PDB import PDBParser, PDBIO, Select
import numpy as np

fold_type = 'GTB'

target_coords = [
        (10.2, 20.5, 30.1), (11.3, 21.7, 31.2), (12.1, 22.9, 32.3),
        (13.4, 23.8, 33.4), (14.5, 24.6, 34.5), (15.7, 25.4, 35.6),
        (16.8, 26.2, 36.7), (17.9, 27.1, 37.8), (18.1, 28.3, 38.9),
        (19.2, 29.5, 39.0)
    ]

if fold_type == 'GTA':
    target_coords = np.array([[ 1.603, 19.007, 10.355], [-0.716, 19.148, 10.857], [ 4.897, 18.192, 11.585], [ 3.434, 18.433, 11.889],
                              [ 4.877, 17.968, 10.078], [ 2.986, 19.226, 10.672], [ 4.610, 16.520,  9.690], [ 0.598, 19.410, 11.266],
                              [ 1.254, 18.402,  9.154], [-0.003, 18.158,  8.777], [-1.113, 18.547,  9.672], [ 3.793, 18.777,  9.572],
                              [ 5.658, 19.372, 11.844], [ 3.247, 19.141, 13.097], [ 5.595, 16.103,  8.757], [ 7.677, 14.976,  7.985],
                              [ 0.831, 19.954, 12.347], [ 7.989, 12.795,  6.698], [ 8.490, 12.923,  9.278], [ 7.015, 14.821, 10.445],
                              [ 7.877, 17.085,  9.421], [-2.280, 18.338,  9.355], [ 8.487, 13.560,  7.905], [ 7.110, 15.788,  9.285],
                              [12.831, 13.363,  7.049], [13.291, 14.698,  6.461], [11.604, 12.848,  6.296], [12.121, 15.688,  6.408],
                              [10.510, 13.918,  6.253], [12.497, 16.997,  5.719], [11.014, 15.129,  5.683], [ 9.977, 14.128,  7.560],
                              [13.877, 12.400,  6.955], [13.783, 14.492,  5.135], [11.093, 11.677,  6.924], [13.052, 17.872,  6.684]])
elif fold_type == 'GTB':
    target_coords = np.array([[ 0.297,-10.026, -4.534], [ 1.190,-11.657, -5.720], [ 2.245, -9.736, -3.099], [ 4.071,-11.142, -3.509],
                              [ 4.264, -9.568, -1.751], [-0.564, -8.645, -2.688], [-0.287, -7.148, -2.837], [-0.679, -9.071, -4.101],
                              [-1.062, -6.875, -4.110], [-0.659, -5.560, -4.737], [ 1.542,-10.285, -4.082], [ 0.099,-10.888, -5.550],
                              [ 2.123,-11.301, -4.806], [ 3.402,-11.734, -4.516], [ 3.494,-10.141, -2.797], [-0.670, -7.937, -4.909],
                              [-1.792, -8.725, -1.997], [-0.746, -6.425, -1.758], [-1.640, -4.598, -4.574], [-1.563, -0.069, -2.051],
                              [-0.582, -2.255, -3.484], [ 0.211, -3.048, -5.970], [-2.242, -2.264, -5.503], [ 0.374,  0.232, -4.051],
                              [-2.079, -0.374, -4.565], [ 3.932,-12.663, -5.180], [-1.092, -3.046, -4.863], [-0.993, -0.598, -3.527],
                              [-4.035,  0.768, -0.403], [-3.632,  0.106,  0.906], [-3.843, -0.224, -1.509], [-2.203, -0.359,  0.735],
                              [-2.511, -0.911, -1.485], [-1.687, -0.971,  2.039], [-2.136, -1.352, -0.224], [-1.563, -0.069, -2.051],
                              [-5.384,  1.128, -0.393], [-3.642,  1.071,  1.926], [-4.888, -1.150, -1.576], [-0.922, -2.091,  1.736]])

distance_cutoff = 6.0  # 距离阈值（6Å）

base_path = f'/home/admin123/work/GTmining/data/structures/{fold_type}/'
save_base_path = f'/home/admin123/work/GTmining/data/partial_structures/{fold_type}/'
if not os.path.exists(save_base_path):
    os.makedirs(save_base_path)



# ==================== 函数 ====================

class ResidueSelect(Select):
    """自定义筛选类：只保留指定的残基"""
    def __init__(self, selected_residues):
        self.selected_residues = selected_residues  # 存储选中的残基对象

    def accept_residue(self, residue):
        # 判断当前残基是否在选中列表中
        return residue in self.selected_residues

def save_selected_residues(structure, selected_residues, output_pdb):
    """将选中的残基保存为新的PDB文件"""
    io = PDBIO()
    io.set_structure(structure)
    # 使用自定义筛选类保存指定残基
    io.save(output_pdb, ResidueSelect(selected_residues))

# ==================== 函数 ====================

for family in os.listdir(base_path):
    # family = 'GT95_aln_check'
    family_path = os.path.join(base_path, family)
    family_save_path = os.path.join(save_base_path, family)
    if not os.path.exists(family_save_path):
        os.makedirs(family_save_path)
    for file in tqdm(os.listdir(family_path), desc=f"Processing {family} families"):
        if file.endswith('.pdb'):
            pdb_file = os.path.join(family_path, file)
            output_pdb = os.path.join(family_save_path, file)

            # 1. 解析PDB文件
            parser = PDBParser(QUIET=True)  # QUIET=True关闭冗余输出
            structure = parser.get_structure("target", pdb_file)

            # 3. 遍历所有原子，计算距离并筛选残基
            selected_residues = set()  # 用集合去重

            for model in structure:
                for chain in model:
                    for residue in chain:
                        # 跳过非标准氨基酸（如溶剂、配体）
                        if residue.id[0] != " ":
                            print(f"Skipping non-standard residue: {residue.get_resname()} in {pdb_file}")
                            continue
                        
                        # 获取残基中所有原子的坐标
                        for atom in residue:
                            atom_coord = np.array(atom.get_coord())  # 原子坐标 (x,y,z)
                            
                            # 计算该原子与所有目标坐标的最小距离
                            distances = np.linalg.norm(target_coords - atom_coord, axis=1)
                            min_distance = np.min(distances)
                            
                            # 如果最小距离≤阈值，选中该残基
                            if min_distance <= distance_cutoff:
                                selected_residues.add(residue)
                                break  # 只要残基有一个原子满足，就无需检查其他原子

            selected_residues = list(selected_residues)  # 转回列表以便后续处理

            # # 3. 输出筛选结果
            # print(f"找到 {len(selected_residues)} 个符合条件的残基：")
            # for res in selected_residues:
            #     chain_id = res.get_parent().id  # 链ID
            #     res_id = res.id[1]              # 残基编号
            #     res_name = res.get_resname()    # 残基名称
            #     print(f"链 {chain_id} | 残基 {res_name} | 编号 {res_id}")

            save_selected_residues(structure, selected_residues, output_pdb)




            # print(pdb_file)
        
        # break

    # break







Processing GT18_aln_check families: 100%|██████████| 11/11 [00:01<00:00, 10.14it/s]


## 评估验证集测试集和训练集的序列相似性

In [42]:
from Bio import Align
from Bio.Seq import Seq
import Bio
from Bio import SeqIO
import multiprocessing
from tqdm import tqdm
from multiprocessing import Pool
import pandas as pd
import os
from collections import defaultdict

fold_type = 'GTA'

for fold in range(1, 11):
    # Read the file and split sequences into two dictionaries
    test = {}
    train_validation = {}
    temp_dict = {}

    for record in SeqIO.parse(f"./predict_data_partial_structures/{fold_type}.fasta", "fasta"):
        temp_dict[record.id] = str(record.seq)

    df = pd.read_excel(f'./predict_data_partial_structures/cluster/{fold_type}_alldata/dataseat_split_{fold}.xlsx')
    for i in range(0, df.shape[0]):
        id_ = df['GenBank'][i]
        split = df['Dataset'][i]
        if split == 'test':
            test[id_] = temp_dict[id_]
        else:
            train_validation[id_] = temp_dict[id_]

    usalign_bash = open(f'./predict_data_partial_structures/{fold_type}_usalign_bash.sh', 'w')
    exe_path = "./exe/USalign"
    save_folder = f"./predict_data_partial_structures/{fold_type}_usalign_results/"
    if not os.path.exists(save_folder):
        os.makedirs(save_folder)
    ecosp_id_list = list(test.keys())

    for ecosp_id, seq in tqdm(test.items()):
        test_family = ecosp_id.split('_')[0]
        test_name = ecosp_id + ".pdb"
        test_file = f"../../data/partial_structures/{fold_type}/{test_family}_aln_check/{test_name}"

        total = len(train_validation)
        for idx, (gtmining_id, gtmining_seq) in enumerate(train_validation.items()):
            gtmining_family = gtmining_id.split('_')[0]
            gtmining_name = gtmining_id + ".pdb"
            gtmining_file = f"../../data/partial_structures/{fold_type}/{gtmining_family}_aln_check/{gtmining_name}"
            out_file = f"{ecosp_id}--{gtmining_id}.txt"

            if test_family != gtmining_family:
                continue

            command = f"{exe_path} {test_file} {gtmining_file} -a T > ./{fold_type}_usalign_results/{out_file}"
            usalign_bash.write(command + f"; echo 'Finished {out_file}. {idx}/{total}'\n")
        # break

    usalign_bash.close()
    


    break

# parallel --jobs 180  < GTA_usalign_bash.sh


100%|██████████| 1316/1316 [00:12<00:00, 101.51it/s]


In [3]:
# 整理分析结果，有400多万条，么逼啊
import os
from collections import defaultdict
from tqdm import tqdm
import pandas as pd
import time

fold_type = 'GTA'
identity_values = {}
chunk_prefix = 'chunk_020'

for file in tqdm(os.listdir(f"./predict_data_partial_structures/{fold_type}_usalign_results/{chunk_prefix}/"), desc="Processing USalign results"):
    # time.sleep(0.001)
    ecosp_id = file.split('--')[0]
    if file.endswith('.txt'):
        with open(os.path.join(f'./predict_data_partial_structures/{fold_type}_usalign_results/{chunk_prefix}/', file), 'r') as f:
            lines = f.readlines()
            for line in lines:
                if line.startswith("TM-score= ") and "if normalized by average length of two structures" in line:
                    identity = line.split("TM-score= ")[1].split(" ")[0]
                    if ecosp_id in identity_values:
                        if identity > identity_values[ecosp_id]:
                            identity_values[ecosp_id] = identity
                    else:
                        identity_values[ecosp_id] = identity
                    break


df = pd.DataFrame(
    identity_values.items(),
    columns=["test_id", "max_identity"]
)


df.to_excel(f"./predict_data_partial_structures/{fold_type}_usalign_results/{chunk_prefix}.xlsx", index=False)


Processing USalign results: 100%|██████████| 232086/232086 [18:59<00:00, 203.74it/s]


## 结合准确率和相似性进行分析

In [7]:
# 合并上述的表格成一个

import pandas as pd
import os

# ====================== 你只需要改这里 ======================
# 你的文件夹路径（和你保存时的路径完全一样）
base_dir = "./predict_data_partial_structures"
fold_type = "GTA"  # 比如 train/val/test，改成你真实的
# ==========================================================

# 拼接完整文件夹路径
folder_path = f"{base_dir}/{fold_type}_usalign_results/"

# 1. 获取文件夹里所有 .xlsx 文件
excel_files = [f for f in os.listdir(folder_path) if f.endswith(".xlsx")]

# 2. 逐个读取并存储到列表
df_list = []
for file in excel_files:
    file_path = os.path.join(folder_path, file)
    df = pd.read_excel(file_path)
    df_list.append(df)
    print(f"已读取：{file}，行数：{len(df)}")

# 3. 合并所有表格
df_combined = pd.concat(df_list, ignore_index=True)

# 4. 保存合并后的大表
output_path = os.path.join(folder_path, f"{fold_type}_usalign_results_combined.xlsx")
df_combined.to_excel(output_path, index=False)

print("\n✅ 合并完成！")
print(f"总文件数：{len(excel_files)}")
print(f"合并后总行数：{len(df_combined)}")
print(f"保存路径：{output_path}")





已读取：chunk_017.xlsx，行数：45
已读取：chunk_016.xlsx，行数：46
已读取：chunk_003.xlsx，行数：1300
已读取：chunk_015.xlsx，行数：45
已读取：chunk_014.xlsx，行数：46
已读取：chunk_010.xlsx，行数：45
已读取：chunk_002.xlsx，行数：1302
已读取：chunk_007.xlsx，行数：46
已读取：chunk_005.xlsx，行数：46
已读取：chunk_011.xlsx，行数：46
已读取：chunk_004.xlsx，行数：46
已读取：chunk_012.xlsx，行数：46
已读取：chunk_001.xlsx，行数：243
已读取：GTA_usalign_results_combined.xlsx，行数：3935
已读取：chunk_018.xlsx，行数：46
已读取：chunk_019.xlsx，行数：46
已读取：chunk_020.xlsx，行数：360
已读取：chunk_013.xlsx，行数：45
已读取：chunk_009.xlsx，行数：46
已读取：chunk_008.xlsx，行数：45
已读取：chunk_006.xlsx，行数：45

✅ 合并完成！
总文件数：21
合并后总行数：7870
保存路径：./predict_data_partial_structures/GTA_usalign_results/GTA_usalign_results_combined.xlsx


In [8]:
import pandas as pd
from collections import defaultdict

fold_tyep = 'GTA'
result_true = defaultdict(int)
result_false = defaultdict(int)

accuracy_dict = defaultdict(bool)
for fold in range(1, 11):
    df_accuracy = pd.read_excel(f'./result/alldata_itemID_accuracy/{fold_tyep}_fold{fold}_test.xlsx')
    for i in range(0, df_accuracy.shape[0]):
        protein_id = df_accuracy['protein_id'][i]
        predict_label = df_accuracy['predict_label_list'][i]
        true_label = df_accuracy['true_label_list'][i]
        if predict_label == true_label:
            accuracy_dict[protein_id] = True
        else:
            accuracy_dict[protein_id] = False

df_identify = pd.read_excel(f'./predict_data_partial_structures/GTA_usalign_results/{fold_type}_usalign_results_combined.xlsx')
for i in range(0, df_identify.shape[0]):
    protein_id = df_identify['test_id'][i]
    max_identity = df_identify['max_identity'][i]
    if max_identity <= 0.1:
        if accuracy_dict[protein_id]:
            result_true['0_10'] += 1
        else:
            result_false['0_10'] += 1
    elif max_identity <= 0.2:
        if accuracy_dict[protein_id]:
            result_true['10_20'] += 1
        else:
            result_false['10_20'] += 1
    elif max_identity <= 0.3:
        if accuracy_dict[protein_id]:
            result_true['20_30'] += 1
        else:
            result_false['20_30'] += 1
    elif max_identity <= 0.4:
        if accuracy_dict[protein_id]:
            result_true['30_40'] += 1
        else:
            result_false['30_40'] += 1
    elif max_identity <= 0.5:
        if accuracy_dict[protein_id]:
            result_true['40_50'] += 1
        else:
            result_false['40_50'] += 1
    elif max_identity <= 0.6:
        if accuracy_dict[protein_id]:
            result_true['50_60'] += 1
        else:
            result_false['50_60'] += 1
    elif max_identity <= 0.7:
        if accuracy_dict[protein_id]:
            result_true['60_70'] += 1
        else:
            result_false['60_70'] += 1
    elif max_identity <= 0.8:
        if accuracy_dict[protein_id]:
            result_true['70_80'] += 1
        else:
            result_false['70_80'] += 1
    elif max_identity <= 0.9:
        if accuracy_dict[protein_id]:
            result_true['80_90'] += 1
        else:
            result_false['80_90'] += 1
    elif max_identity <= 1.0:
        if accuracy_dict[protein_id]:
            result_true['90_100'] += 1
        else:
            result_false['90_100'] += 1



    # break

for key in result_true.keys():
    print(f"Identity range {key}: True count = {result_true[key]}, False count = {result_false[key]}, Accuracy = {result_true[key]/(result_true[key]+result_false[key]):.4f}")





Identity range 60_70: True count = 1156, False count = 380, Accuracy = 0.7526
Identity range 70_80: True count = 1414, False count = 398, Accuracy = 0.7804
Identity range 80_90: True count = 1294, False count = 298, Accuracy = 0.8128
Identity range 90_100: True count = 376, False count = 56, Accuracy = 0.8704
Identity range 50_60: True count = 614, False count = 368, Accuracy = 0.6253
Identity range 40_50: True count = 386, False count = 468, Accuracy = 0.4520
Identity range 30_40: True count = 150, False count = 428, Accuracy = 0.2595
Identity range 20_30: True count = 30, False count = 52, Accuracy = 0.3659
Identity range 0_10: True count = 2, False count = 0, Accuracy = 1.0000


# Glydentity-计算train和test的序列相似性

In [4]:
'''
下面是代码
'''

from Bio import Align
from Bio.Seq import Seq
import Bio
from Bio import SeqIO
import multiprocessing
from tqdm import tqdm
from multiprocessing import Pool
import pandas as pd

def calculate_protein_similarity(seq1, seq2):
    """
    计算两条蛋白质序列的相似性（兼容不同Biopython版本）
    :param seq1: 第一条蛋白质序列（字符串/Seq对象）
    :param seq2: 第二条蛋白质序列（字符串/Seq对象）
    :return: 比对结果、一致性比例、相似性比例
    """
    # 1. 初始化比对器，设置为蛋白质序列比对
    aligner = Align.PairwiseAligner()
    aligner.mode = 'global'  # 全局比对
    aligner.substitution_matrix = Align.substitution_matrices.load("BLOSUM62")
    # 用默认缺口罚分（无需手动设置）

    # 2. 执行序列比对，取最优比对结果
    alignments = aligner.align(seq1, seq2)
    best_aln = alignments[0]

    # 3. 兼容不同版本的序列提取方式（核心修复点）
    # 将原始序列转为字符串，方便后续比对
    seq1_str = str(seq1)
    seq2_str = str(seq2)
    
    # 从Alignment对象中提取比对的位置信息
    # best_aln.aligned[0] 是第一条序列的比对区间，best_aln.aligned[1] 是第二条的
    aln_pos1 = best_aln.aligned[0]
    aln_pos2 = best_aln.aligned[1]
    
    # 构建比对后的序列（补全缺口'-'）
    aln_seq1 = []
    aln_seq2 = []
    pos1 = pos2 = 0  # 原始序列的指针
    
    for (start1, end1), (start2, end2) in zip(aln_pos1, aln_pos2):
        # 处理前面的缺口
        gap1 = start1 - pos1
        gap2 = start2 - pos2
        if gap1 > 0:
            aln_seq1.extend(['-'] * gap1)
            aln_seq2.extend(seq2_str[pos2:pos2+gap1])
            pos2 += gap1
        if gap2 > 0:
            aln_seq2.extend(['-'] * gap2)
            aln_seq1.extend(seq1_str[pos1:pos1+gap2])
            pos1 += gap2
        
        # 添加匹配的序列片段
        aln_seq1.extend(seq1_str[start1:end1])
        aln_seq2.extend(seq2_str[start2:end2])
        pos1, pos2 = end1, end2
    
    # 处理末尾的缺口
    if pos1 < len(seq1_str):
        aln_seq1.extend(seq1_str[pos1:])
        aln_seq2.extend(['-'] * (len(seq1_str) - pos1))
    if pos2 < len(seq2_str):
        aln_seq2.extend(seq2_str[pos2:])
        aln_seq1.extend(['-'] * (len(seq2_str) - pos2))
    
    # 转为字符串
    aln_seq1 = ''.join(aln_seq1)
    aln_seq2 = ''.join(aln_seq2)
    
    # 4. 计算一致性（Identity）
    match_count = sum(1 for a, b in zip(aln_seq1, aln_seq2) if a == b and a != '-')
    valid_length = sum(1 for a, b in zip(aln_seq1, aln_seq2) if a != '-' or b != '-')
    identity = match_count / valid_length if valid_length > 0 else 0.0

    # 5. 计算相似性（Similarity）
    blosum62 = Align.substitution_matrices.load("BLOSUM62")
    similarity_count = 0
    for a, b in zip(aln_seq1, aln_seq2):
        if a == '-' or b == '-':
            continue
        if blosum62[a][b] > 0:
            similarity_count += 1
    similarity = similarity_count / valid_length if valid_length > 0 else 0.0

    # 构造易读的比对结果字符串
    aln_result_str = f"{aln_seq1}\n{aln_seq2}\n得分：{best_aln.score:.2f}"
    return aln_result_str, identity, similarity

def a_batch_task(ecosp_id, ecosp_seq, gtmining):
    max_identity = 0
    for gtmining_id, gtmining_seq in gtmining.items():
        _, identity, _ = calculate_protein_similarity(ecosp_seq, gtmining_seq)
        if identity > max_identity:
            max_identity = identity
    return ecosp_id, max_identity

def unpack_batch_task(args):
    """辅助函数：解包参数元组并调用a_batch_task"""
    return a_batch_task(*args)


# ------------------- 测试示例 -------------------
if __name__ == "__main__":

    
    # Read the file and split sequences into two dictionaries
    test = {}
    train_validation = {}

    df_train_validation = pd.read_csv(f'./result/Glyidentity/gtb/train.csv')
    for i in range(0, df_train_validation.shape[0]):
        id_ = df_train_validation['Uniprot'][i]
        seq = df_train_validation['Sequence'][i]
        train_validation[id_] = seq

    df_test = pd.read_csv(f'./result/Glyidentity/gtb/test.csv')
    for i in range(0, df_test.shape[0]):
        id_ = df_test['Uniprot'][i]
        seq = df_test['Sequence'][i]
        test[id_] = seq

    # 构造参数元组列表（每个元组包含两个参数）
    task_args = [(ecosp_id, seq, train_validation) for ecosp_id, seq in test.items()]

    # 创建进程池执行任务（用具名函数替代lambda）
    with multiprocessing.Pool(processes=180) as pool:
        # 使用具名的unpack_batch_task函数，可被pickle序列化
        identitys = list(tqdm(
            pool.imap(unpack_batch_task, task_args),  # 替换lambda为具名函数
            total=len(test),
            desc="计算序列相似性"  # 可选：给进度条加描述，更清晰
        ))

    df = pd.DataFrame({
        "test_id": [item[0] for item in identitys],
        "max_identity": [item[1] for item in identitys]
    })

    df.to_excel(f"./result/Glyidentity/gtb.xlsx", index=False)






计算序列相似性: 100%|██████████| 281/281 [08:27<00:00,  1.81s/it]  


# 提取用来比较不同模型的数据-Fold5

In [ ]:
import pandas as pd
import os
import shutil
from tqdm import tqdm

df = pd.read_excel('../data/cluster/GTB_alldata/dataseat_split_5.xlsx')
df_test = df[df['Dataset'] == 'test']
df_test.reset_index(inplace=True, drop=True)

storage_fold = './predict_data/GTB_test/'
if not os.path.exists(storage_fold):
    os.makedirs(storage_fold)

pdb_fold = '../data/structures/GTB/'

for i in tqdm(range(0, df_test.shape[0])):
    sub_fold = f"{df_test['Family'][i]}_aln_check"
    file_name = f"{df_test['GenBank'][i]}.pdb"

    # 构建源文件路径
    source_path = os.path.join(pdb_fold, sub_fold, file_name)
    # 构建目标文件路径
    destination_path = os.path.join(storage_fold, file_name)
    
    # 检查源文件是否存在，然后复制
    if os.path.exists(source_path):
        shutil.copy2(source_path, destination_path)
    else:
        print(f"Source file does not exist: {source_path}")


# export HF_ENDPOINT=https://hf-mirror.com


100%|██████████| 2135/2135 [00:36<00:00, 58.92it/s]


# 评估别人的实验结果在我们的模型上表现如何

## 序列相似性-Max Identify

In [ ]:
'''
下面是代码
'''

from Bio import Align
from Bio.Seq import Seq
import Bio
from Bio import SeqIO
import multiprocessing
from tqdm import tqdm
from multiprocessing import Pool
import pandas as pd

def calculate_protein_similarity(seq1, seq2):
    """
    计算两条蛋白质序列的相似性（兼容不同Biopython版本）
    :param seq1: 第一条蛋白质序列（字符串/Seq对象）
    :param seq2: 第二条蛋白质序列（字符串/Seq对象）
    :return: 比对结果、一致性比例、相似性比例
    """
    # 1. 初始化比对器，设置为蛋白质序列比对
    aligner = Align.PairwiseAligner()
    aligner.mode = 'global'  # 全局比对
    aligner.substitution_matrix = Align.substitution_matrices.load("BLOSUM62")
    # 用默认缺口罚分（无需手动设置）

    # 2. 执行序列比对，取最优比对结果
    alignments = aligner.align(seq1, seq2)
    best_aln = alignments[0]

    # 3. 兼容不同版本的序列提取方式（核心修复点）
    # 将原始序列转为字符串，方便后续比对
    seq1_str = str(seq1)
    seq2_str = str(seq2)
    
    # 从Alignment对象中提取比对的位置信息
    # best_aln.aligned[0] 是第一条序列的比对区间，best_aln.aligned[1] 是第二条的
    aln_pos1 = best_aln.aligned[0]
    aln_pos2 = best_aln.aligned[1]
    
    # 构建比对后的序列（补全缺口'-'）
    aln_seq1 = []
    aln_seq2 = []
    pos1 = pos2 = 0  # 原始序列的指针
    
    for (start1, end1), (start2, end2) in zip(aln_pos1, aln_pos2):
        # 处理前面的缺口
        gap1 = start1 - pos1
        gap2 = start2 - pos2
        if gap1 > 0:
            aln_seq1.extend(['-'] * gap1)
            aln_seq2.extend(seq2_str[pos2:pos2+gap1])
            pos2 += gap1
        if gap2 > 0:
            aln_seq2.extend(['-'] * gap2)
            aln_seq1.extend(seq1_str[pos1:pos1+gap2])
            pos1 += gap2
        
        # 添加匹配的序列片段
        aln_seq1.extend(seq1_str[start1:end1])
        aln_seq2.extend(seq2_str[start2:end2])
        pos1, pos2 = end1, end2
    
    # 处理末尾的缺口
    if pos1 < len(seq1_str):
        aln_seq1.extend(seq1_str[pos1:])
        aln_seq2.extend(['-'] * (len(seq1_str) - pos1))
    if pos2 < len(seq2_str):
        aln_seq2.extend(seq2_str[pos2:])
        aln_seq1.extend(['-'] * (len(seq2_str) - pos2))
    
    # 转为字符串
    aln_seq1 = ''.join(aln_seq1)
    aln_seq2 = ''.join(aln_seq2)
    
    # 4. 计算一致性（Identity）
    match_count = sum(1 for a, b in zip(aln_seq1, aln_seq2) if a == b and a != '-')
    valid_length = sum(1 for a, b in zip(aln_seq1, aln_seq2) if a != '-' or b != '-')
    identity = match_count / valid_length if valid_length > 0 else 0.0

    # 5. 计算相似性（Similarity）
    blosum62 = Align.substitution_matrices.load("BLOSUM62")
    similarity_count = 0
    for a, b in zip(aln_seq1, aln_seq2):
        if a == '-' or b == '-':
            continue
        if blosum62[a][b] > 0:
            similarity_count += 1
    similarity = similarity_count / valid_length if valid_length > 0 else 0.0

    # 构造易读的比对结果字符串
    aln_result_str = f"{aln_seq1}\n{aln_seq2}\n得分：{best_aln.score:.2f}"
    return aln_result_str, identity, similarity

def a_batch_task(ecosp_id, ecosp_seq, gtmining):
    max_identity = 0
    for gtmining_id, gtmining_seq in gtmining.items():
        _, identity, _ = calculate_protein_similarity(ecosp_seq, gtmining_seq)
        if identity > max_identity:
            max_identity = identity
    return ecosp_id, max_identity

def unpack_batch_task(args):
    """辅助函数：解包参数元组并调用a_batch_task"""
    return a_batch_task(*args)


# ------------------- 测试示例 -------------------
if __name__ == "__main__":
    dataset_dict = {}
    for record in SeqIO.parse("./result/GTB.fasta", "fasta"):
    # for record in SeqIO.parse("./result/GTA.fasta", "fasta"):
        dataset_dict[record.id] = str(record.seq)

    query_dict = {}
    for record in SeqIO.parse("./result/other_GTB.fasta", "fasta"):
    # for record in SeqIO.parse("./result/jk.txt", "fasta"):
        query_dict[record.id] = str(record.seq)


    # 构造参数元组列表（每个元组包含两个参数）
    task_args = [(query_id, seq, dataset_dict) for query_id, seq in query_dict.items()]

    # 创建进程池执行任务（用具名函数替代lambda）
    with multiprocessing.Pool(processes=150) as pool:
        # 使用具名的unpack_batch_task函数，可被pickle序列化
        identitys = list(tqdm(
            pool.imap(unpack_batch_task, task_args),  # 替换lambda为具名函数
            total=len(query_dict),
            desc="计算序列相似性"  # 可选：给进度条加描述，更清晰
        ))

    df = pd.DataFrame({
        "query_id": [item[0] for item in identitys],
        "max_identity": [item[1] for item in identitys]
    })

    df.to_excel(f"./result/other_gtb.xlsx", index=False)



计算序列相似性: 100%|██████████| 4/4 [00:49<00:00, 12.44s/it]


## 结构相似性-TMscore

### 提取局部结构

In [5]:
# GTA
# 路径：/home/admin123/work/GTmining/data/structures/GTA/
# GTB
# 路径：/home/admin123/work/GTmining/data/structures/GTB/

import os
from tqdm import tqdm
from Bio.PDB import PDBParser, PDBIO, Select
import numpy as np


fold_type = 'GTB'


base_path = f'/home/admin123/work/GTmining/diffpool/result/other_{fold_type}_jk/'
save_base_path = f'/home/admin123/work/GTmining/diffpool/result/other_{fold_type}_partial_jk/'
if not os.path.exists(save_base_path):
    os.makedirs(save_base_path)

if fold_type == 'GTA':
    target_coords = np.array([[ 1.603, 19.007, 10.355], [-0.716, 19.148, 10.857], [ 4.897, 18.192, 11.585], [ 3.434, 18.433, 11.889],
                              [ 4.877, 17.968, 10.078], [ 2.986, 19.226, 10.672], [ 4.610, 16.520,  9.690], [ 0.598, 19.410, 11.266],
                              [ 1.254, 18.402,  9.154], [-0.003, 18.158,  8.777], [-1.113, 18.547,  9.672], [ 3.793, 18.777,  9.572],
                              [ 5.658, 19.372, 11.844], [ 3.247, 19.141, 13.097], [ 5.595, 16.103,  8.757], [ 7.677, 14.976,  7.985],
                              [ 0.831, 19.954, 12.347], [ 7.989, 12.795,  6.698], [ 8.490, 12.923,  9.278], [ 7.015, 14.821, 10.445],
                              [ 7.877, 17.085,  9.421], [-2.280, 18.338,  9.355], [ 8.487, 13.560,  7.905], [ 7.110, 15.788,  9.285],
                              [12.831, 13.363,  7.049], [13.291, 14.698,  6.461], [11.604, 12.848,  6.296], [12.121, 15.688,  6.408],
                              [10.510, 13.918,  6.253], [12.497, 16.997,  5.719], [11.014, 15.129,  5.683], [ 9.977, 14.128,  7.560],
                              [13.877, 12.400,  6.955], [13.783, 14.492,  5.135], [11.093, 11.677,  6.924], [13.052, 17.872,  6.684]])
elif fold_type == 'GTB':
    target_coords = np.array([[ 0.297,-10.026, -4.534], [ 1.190,-11.657, -5.720], [ 2.245, -9.736, -3.099], [ 4.071,-11.142, -3.509],
                              [ 4.264, -9.568, -1.751], [-0.564, -8.645, -2.688], [-0.287, -7.148, -2.837], [-0.679, -9.071, -4.101],
                              [-1.062, -6.875, -4.110], [-0.659, -5.560, -4.737], [ 1.542,-10.285, -4.082], [ 0.099,-10.888, -5.550],
                              [ 2.123,-11.301, -4.806], [ 3.402,-11.734, -4.516], [ 3.494,-10.141, -2.797], [-0.670, -7.937, -4.909],
                              [-1.792, -8.725, -1.997], [-0.746, -6.425, -1.758], [-1.640, -4.598, -4.574], [-1.563, -0.069, -2.051],
                              [-0.582, -2.255, -3.484], [ 0.211, -3.048, -5.970], [-2.242, -2.264, -5.503], [ 0.374,  0.232, -4.051],
                              [-2.079, -0.374, -4.565], [ 3.932,-12.663, -5.180], [-1.092, -3.046, -4.863], [-0.993, -0.598, -3.527],
                              [-4.035,  0.768, -0.403], [-3.632,  0.106,  0.906], [-3.843, -0.224, -1.509], [-2.203, -0.359,  0.735],
                              [-2.511, -0.911, -1.485], [-1.687, -0.971,  2.039], [-2.136, -1.352, -0.224], [-1.563, -0.069, -2.051],
                              [-5.384,  1.128, -0.393], [-3.642,  1.071,  1.926], [-4.888, -1.150, -1.576], [-0.922, -2.091,  1.736]])

distance_cutoff = 6.0  # 距离阈值（6Å）

# ==================== 函数 ====================

class ResidueSelect(Select):
    """自定义筛选类：只保留指定的残基"""
    def __init__(self, selected_residues):
        self.selected_residues = selected_residues  # 存储选中的残基对象

    def accept_residue(self, residue):
        # 判断当前残基是否在选中列表中
        return residue in self.selected_residues

def save_selected_residues(structure, selected_residues, output_pdb):
    """将选中的残基保存为新的PDB文件"""
    io = PDBIO()
    io.set_structure(structure)
    # 使用自定义筛选类保存指定残基
    io.save(output_pdb, ResidueSelect(selected_residues))

# ==================== 函数 ====================



family_path = base_path
family_save_path = save_base_path

for file in tqdm(os.listdir(family_path), desc=f"Processing families"):
    if file.endswith('.pdb'):
        pdb_file = os.path.join(family_path, file)
        output_pdb = os.path.join(family_save_path, file)

        # 1. 解析PDB文件
        parser = PDBParser(QUIET=True)  # QUIET=True关闭冗余输出
        structure = parser.get_structure("target", pdb_file)

        # 3. 遍历所有原子，计算距离并筛选残基
        selected_residues = set()  # 用集合去重

        for model in structure:
            for chain in model:
                for residue in chain:
                    # 跳过非标准氨基酸（如溶剂、配体）
                    if residue.id[0] != " ":
                        print(f"Skipping non-standard residue: {residue.get_resname()} in {pdb_file}")
                        continue
                    
                    # 获取残基中所有原子的坐标
                    for atom in residue:
                        atom_coord = np.array(atom.get_coord())  # 原子坐标 (x,y,z)
                        
                        # 计算该原子与所有目标坐标的最小距离
                        distances = np.linalg.norm(target_coords - atom_coord, axis=1)
                        min_distance = np.min(distances)
                        
                        # 如果最小距离≤阈值，选中该残基
                        if min_distance <= distance_cutoff:
                            selected_residues.add(residue)
                            break  # 只要残基有一个原子满足，就无需检查其他原子

        selected_residues = list(selected_residues)  # 转回列表以便后续处理

        # # 3. 输出筛选结果
        # print(f"找到 {len(selected_residues)} 个符合条件的残基：")
        # for res in selected_residues:
        #     chain_id = res.get_parent().id  # 链ID
        #     res_id = res.id[1]              # 残基编号
        #     res_name = res.get_resname()    # 残基名称
        #     print(f"链 {chain_id} | 残基 {res_name} | 编号 {res_id}")

        save_selected_residues(structure, selected_residues, output_pdb)




            # print(pdb_file)
        
        # break

    # break







Processing families: 100%|██████████| 1/1 [00:00<00:00, 12.20it/s]


### 计算TMsocre-用USAlign

In [6]:
from Bio import Align
from Bio.Seq import Seq
import Bio
from Bio import SeqIO
import multiprocessing
from tqdm import tqdm
from multiprocessing import Pool
import pandas as pd
import os
from collections import defaultdict


fold_type = 'GTB'
usalign_bash = open(f'./result/{fold_type}_usalign_bash.sh', 'w')
exe_path = "../predict_data/exe/USalign"
save_folder = f"./result/other_{fold_type}_usalign_results_jk/"
if not os.path.exists(save_folder):
    os.makedirs(save_folder)

for other_file in os.listdir(f"./result/other_{fold_type}_partial_jk/"):
    if not other_file.endswith('.pdb'):
        continue
    other_id = other_file.split('.')[0]
    other_file_file = f"./other_{fold_type}_partial_jk/{other_file}"

    for family_fold in os.listdir(f"../data/partial_structures/{fold_type}"):
        total = len(os.listdir(f"../data/partial_structures/{fold_type}/{family_fold}/"))
        idx = 0
        for file in os.listdir(f"../data/partial_structures/{fold_type}/{family_fold}/"):
            idx += 1
            if not file.endswith('.pdb'):
                continue

            # print(f"Comparing {other_file} with {file}")
            file_id = file.split('.')[0]
            file_file = f"../../data/partial_structures/{fold_type}/{family_fold}/{file}"
            out_file = f"{other_id}--{file_id}.txt"


            command = f"{exe_path} {other_file_file} {file_file} -a T > ./other_{fold_type}_usalign_results_jk/{out_file}"
            usalign_bash.write(command + f"; echo 'Finished {out_file}. {idx}/{total}'\n")

            # break
        # break
    # break

usalign_bash.close()

# parallel --jobs 180  < GTA_usalign_bash.sh
# parallel --jobs 180  < GTB_usalign_bash.sh


### 汇总TM-Score

In [7]:
# 整理分析结果，有400多万条，么逼啊
import os
from collections import defaultdict
from tqdm import tqdm
import pandas as pd
import time

fold_type = 'GTB'
identity_values = {}
max_identity_id = {}

for file in tqdm(os.listdir(f"./result/other_{fold_type}_usalign_results_jk/"), desc="Processing USalign results"):
    # time.sleep(0.001)
    file_id = file.split('--')[0]
    target_id = file.split('--')[1].split('.txt')[0]
    if not file.endswith('.txt'):
        continue

    with open(os.path.join(f'./result/other_{fold_type}_usalign_results_jk/', file), 'r') as f:
        lines = f.readlines()
        for line in lines:
            if line.startswith("TM-score= ") and "if normalized by average length of two structures" in line:
                identity = line.split("TM-score= ")[1].split(" ")[0]
                if file_id in identity_values:
                    if identity > identity_values[file_id]:
                        identity_values[file_id] = identity
                        max_identity_id[file_id] = target_id
                else:
                    identity_values[file_id] = identity
                    max_identity_id[file_id] = target_id
                break

    # break


df = pd.DataFrame(
    identity_values.items(),
    columns=["test_id", "max_identity"]
)

df.to_excel(f"./result/other_{fold_type}_usalign_results_jk.xlsx", index=False)


df = pd.DataFrame(
    max_identity_id.items(),
    columns=["test_id", "target_id"]
)

df.to_excel(f"./result/other_{fold_type}_usalign_results_target_jk.xlsx", index=False)


Processing USalign results: 100%|██████████| 44491/44491 [00:01<00:00, 31261.38it/s]
